# Information and liquidity shocks

## Summary

How do spreads and depth change around scheduled kickoff and receipt of scoreboard updates?

The default workspace is **synthetic**. Its outputs establish software behavior, not empirical market findings. Run `pml demo` first, then execute this notebook top to bottom. Observed data can replace the workspace without changing the calculation functions.

## Context and methods

A scoreboard update is an observed message, not the instant of a goal. Corrections remain separate anchors. Compare 60-second windows around an anchor with a mechanically selected local period three minutes earlier. This is descriptive, not a causal estimate.

### Key assumptions

No observed edge is claimed. Unusable books, unknown costs and unavailable labels must stay visible. Results below are derived or simulated; inspect the input provenance before interpreting them.

## Data

The immutable run manifest identifies the exact captured inputs. Set `PML_WORKSPACE` to a workspace with the same layout to analyze another capture.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
import plotly.express as px

from prediction_market_lab.research import load_table, shock_study
from prediction_market_lab.storage import verify_run

workspace = Path(os.environ.get("PML_WORKSPACE", "data/demo")).resolve()
descriptor = json.loads((workspace / "workspace.json").read_text())
run = workspace / descriptor["run"]
manifest = verify_run(run)
features = load_table(run, "features")
provenance = sorted(features.input_provenance.unique())
print(
	{
		"input_provenance": provenance,
		"input_sha256": manifest["input_sha256"],
		"feature_rows": len(features),
	}
)
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = ["#315c8c", "#b9812b", "#78619e"]

## Results

These calculations call the same tested functions as the terminal and CLI reports.

In [ ]:
events = load_table(run, "events")
study = shock_study(features, events)
episodes = pd.DataFrame(study["episodes"])
episodes.head(12)

In [ ]:
fig = px.scatter(
	episodes,
	x="spread_change_pp",
	y="bid_depth_change_shares",
	color="anchor",
	hover_data=["fixture_id", "at"],
	title="Liquidity changes around received information",
	labels={
		"spread_change_pp": "After minus before spread (pp)",
		"bid_depth_change_shares": "After minus before bid depth (shares)",
	},
)
fig.add_vline(x=0, line_color="#777777")
fig.add_hline(y=0, line_color="#777777")
fig.show()

## Validation and limitations

The following tables and checks are part of the result, including insufficient samples and negative findings.

In [ ]:
print(json.dumps(study["event_uncertainty"], indent=2))
print("\n".join(study["limitations"]))

## Takeaways

Use this experiment to inspect measurement, timing and execution assumptions. For synthetic inputs, differences across models or strategies have no external validity. For observed inputs, report the covered events, exclusions, uncertainty and sensitivity before making a market claim.

Method and sources: [research protocol](../docs/research-protocol.md), [data dictionary](../docs/data-dictionary.md), [source register](../docs/sources.md).